In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount("/content/Drive")

Mounted at /content/Drive


In [ ]:
features = pd.read_csv('/content/Drive/My Drive/Features data set.csv')
sales = pd.read_csv('/content/Drive/My Drive/sales data-set.csv')
stores = pd.read_csv('/content/Drive/My Drive/stores data-set.csv')

## 1. Date Preparation

### Business Question
How can we extract useful time components from the sales date for time-based analysis?

In [ ]:
sales['Date'] = pd.to_datetime(
    sales['Date'],dayfirst = True
)
sales['Year'] = sales['Date'].dt.year
sales['Month'] = sales['Date'].dt.month
sales['Month_Name'] = sales['Date'].dt.month_name()
sales['Day_of_Week'] = sales['Date'].dt.day_name()
sales['Week'] = sales['Date'].dt.isocalendar().week

In [ ]:
sales['Month_Name'] = pd.Categorical(
    sales['Month_Name'],
    categories = ['January','February','March','April','May','June',
    'July','August','September','October','November','December'],ordered=True
)

In [ ]:
sales[['Date', 'Year', 'Month', 'Month_Name', 'Day_of_Week','Week']].head()

,Date,Year,Month,Month_Name,Day_of_Week,Week
0,2010-02-05,2010,2,February,Friday,5
1,2010-02-12,2010,2,February,Friday,6
2,2010-02-19,2010,2,February,Friday,7
3,2010-02-26,2010,2,February,Friday,8
4,2010-03-05,2010,3,March,Friday,9


### Insight
Date components such as year, month, and day of week allow us to analyze sales patterns across different time periods.

## 2. Total Sales by Year

### Business Question
How does total sales performance vary across different years?

In [ ]:
yearly_sales = sales.groupby('Year')['Weekly_Sales'].sum()
yearly_sales

,Weekly_Sales
Year,
2010,2.288886e+09
2011,2.448200e+09
2012,2.000133e+09


### Insight
2011 recorded the highest total sales, while 2012 recorded the lowest among the three years.

## 3. Year-over-Year Sales Change

### Business Question
How did total sales change from one year to the next?

In [ ]:
yearly_sales_change = yearly_sales.diff().rename('Sales_Change')
yearly_sales_change

,Sales_Change
Year,
2010,NaN
2011,1.593139e+08
2012,-4.480671e+08


### Insight
Sales increased from 2010 to 2011 but declined substantially from 2011 to 2012, making 2012 the weaker year in terms of year-over-year sales change.

## 4. Monthly Sales Performance

### Business Question
Which months generate the highest and lowest total sales?

In [ ]:
monthly_sales_name = (
    sales.groupby('Month_Name')['Weekly_Sales']
    .sum()
    .sort_values(ascending=False)
)

monthly_sales_name

/tmp/ipykernel_523/3122998706.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sales.groupby('Month_Name')['Weekly_Sales']


,Weekly_Sales
Month_Name,
July,6.500010e+08
April,6.468598e+08
June,6.226299e+08
August,6.130902e+08
March,5.927859e+08
October,5.847848e+08
September,5.787612e+08
December,5.768386e+08
February,5.687279e+08


### Insight
July recorded the highest total sales, while January recorded the lowest total sales among the months.

## 5. Monthly Sales Trend by Year

### Business Question
How do monthly sales change across different years?

In [ ]:
monthly_year_sales = sales.groupby(['Month','Year'])['Weekly_Sales'].sum().unstack()
monthly_year_sales

Year,2010,2011,2012
Month,,,
1,NaN,1.637040e+08,1.688945e+08
2,1.903330e+08,1.863313e+08,1.920636e+08
3,1.819198e+08,1.793564e+08,2.315097e+08
4,2.314124e+08,2.265265e+08,1.889209e+08
5,1.867109e+08,1.816482e+08,1.887665e+08
6,1.922462e+08,1.897734e+08,2.406103e+08
7,2.325801e+08,2.299114e+08,1.875095e+08
8,1.876401e+08,1.885993e+08,2.368508e+08
9,1.772679e+08,2.208477e+08,1.806455e+08


### Insight
Monthly sales patterns vary across years. December recorded high sales in 2010 and 2011, while several months performed differently in 2012. November and December have missing values for 2012.

## 6. Best and Worst Monthly Performance by Year

### Business Question
Which month recorded the highest and lowest sales in each year?

In [ ]:
highest_month_by_year = monthly_year_sales.idxmax()
highest_sales_by_year = monthly_year_sales.max()
lowest_month_by_year = monthly_year_sales.idxmin()
lowest_sales_by_year = monthly_year_sales.min()
highest_month_by_year,highest_sales_by_year,lowest_month_by_year,lowest_sales_by_year

(Year
 2010    12
 2011    12
 2012     6
 dtype: int32,
 Year
 2010    2.887605e+08
 2011    2.880781e+08
 2012    2.406103e+08
 dtype: float64,
 Year
 2010    9
 2011    1
 2012    1
 dtype: int32,
 Year
 2010    1.772679e+08
 2011    1.637040e+08
 2012    1.688945e+08
 dtype: float64)

### Insight
December recorded the highest monthly sales in both 2010 and 2011, while June was the highest in 2012. The lowest sales occurred in September 2010 and January in both 2011 and 2012.

## 7. Average Weekly Sales by Month

### Business Question
Which months have the highest and lowest average weekly sales?

In [ ]:
monthly_avg_sales = sales.groupby('Month_Name',observed=True)['Weekly_Sales'].mean().rename('Avg_Sales')
monthly_avg_sales

,Avg_Sales
Month_Name,
January,14126.075111
February,16008.779217
March,15416.657597
April,15650.338357
May,15776.337202
June,16326.137002
July,15861.419650
August,16062.516933
September,15095.886154


### Insight
December recorded the highest average weekly sales, followed by November, while January recorded the lowest. This indicates that average weekly sales vary across months.

## 8. Weekly Sales by Day of Week

### Business Question
How do sales vary across different days of the week?

In [ ]:
sales['Day_of_Week'].value_counts()

,count
Day_of_Week,
Friday,421570


### Insight
All sales records fall on Fridays, so this dataset does not support a comparison of sales across different days of the week.

## 8. Weekly Sales by Week of the Year

### Business Question
Which weeks of the year have the highest and lowest total sales?

In [ ]:
weekly_sales = sales.groupby(['Week','Year'])['Weekly_Sales'].sum().unstack()
weekly_sales

Year,2010,2011,2012
Week,,,
1,NaN,42775787.77,44955421.95
2,NaN,40673678.04,42023078.48
3,NaN,40654648.03,42080996.56
4,NaN,39599852.99,39834974.67
5,49750740.50,46153111.12,46085608.09
6,48336677.63,47336192.79,50009407.92
7,48276993.78,48716164.12,50197056.96
8,43968571.13,44125859.84,45771506.57
9,46871470.30,46980603.74,46861034.97


In [ ]:
highest_sales_week = sales.loc[sales['Weekly_Sales'].idxmax()]
lowest_sales_week = sales.loc[sales['Weekly_Sales'].idxmin()]

highest_sales_week, lowest_sales_week

(Store                            10
 Dept                             72
 Date            2010-11-26 00:00:00
 Weekly_Sales              693099.36
 IsHoliday                      True
 Year                           2010
 Month                            11
 Month_Name                 November
 Day_of_Week                  Friday
 Week                             47
 Name: 95373, dtype: object,
 Store                            28
 Dept                              6
 Date            2010-10-08 00:00:00
 Weekly_Sales               -4988.94
 IsHoliday                     False
 Year                           2010
 Month                            10
 Month_Name                  October
 Day_of_Week                  Friday
 Week                             40
 Name: 267730, dtype: object)

### Insight
The highest individual weekly sales occurred at Store 10, Department 72, on November 26, 2010, a holiday week. The lowest record was a negative sale at Store 28, Department 6, on October 8, 2010, and should be reviewed to understand the negative value.

## 10. Holiday vs Non-Holiday Sales by Year

### Business Question
How do total sales during holiday and non-holiday weeks compare across different years?

In [ ]:
holiday_year_sales = sales.groupby(['Year','IsHoliday'])['Weekly_Sales'].sum().unstack()
holiday_year_sales

IsHoliday,False,True
Year,,
2010,2.088662e+09,2.002246e+08
2011,2.241465e+09,2.067355e+08
2012,1.901793e+09,9.833947e+07


### Insight
Non-holiday weeks generated higher total sales than holiday weeks in all three years. However, total sales alone cannot establish which group performed better per week because the number of weeks in each group may differ.

## 11. Average Weekly Sales: Holiday vs Non-Holiday

### Business Question
How does average weekly sales differ between holiday and non-holiday weeks across years?

In [ ]:
holiday_year_sales = sales.groupby(['Year','IsHoliday'])['Weekly_Sales'].mean().unstack().round(2)
holiday_year_sales

IsHoliday,False,True
Year,,
2010,16203.11,17005.66
2011,15837.05,17343.58
2012,15656.36,16480.55


### Insight
Average weekly sales were higher during holiday weeks than non-holiday weeks in all three years. This suggests that holiday weeks were associated with stronger average sales, although this comparison alone does not establish that holidays caused the increase.

## 12. Store Sales Trends Over Time

### Business Question
How do total sales for each store change across different years?

In [ ]:
store_year_sales = sales.groupby(['Store','Year'])['Weekly_Sales'].sum().unstack()
store_year_sales

Year,2010,2011,2012
Store,,,
1,7.327883e+07,8.092192e+07,68202058.02
2,9.527786e+07,9.860788e+07,81496695.37
3,1.874542e+07,2.081688e+07,18024439.50
4,9.568047e+07,1.110923e+08,92771189.24
5,1.483603e+07,1.647082e+07,14168838.13
6,7.691232e+07,8.052876e+07,66315047.00
7,2.556808e+07,3.066264e+07,25367556.47
8,4.320447e+07,4.751279e+07,39233920.13
9,2.512922e+07,2.868597e+07,23974029.58


### Insight
Many stores recorded higher total sales in 2011 than in 2010, followed by lower sales in 2012. However, some stores continued to grow in 2012, indicating that sales trends differed across stores.

## 13. Store Sales Change: 2010 to 2012

### Business Question
Which stores experienced the largest sales increase or decrease between 2010 and 2012?

In [ ]:
store_sales_change = (store_year_sales[2012] - store_year_sales[2010]).rename('sales_change')
store_sales_change.sort_values(ascending=False)

,sales_change
Store,
38,2043278.93
44,579854.27
39,103386.53
7,-200521.68
42,-547730.06
5,-667192.64
3,-720979.50
17,-819541.81
41,-850380.72


In [ ]:
store_sales_change = (store_year_sales[2012] - store_year_sales[2010]).rename('sales_change')
store_sales_change.sort_values().head()

,sales_change
Store,
14,-28020844.12
27,-18093120.45
35,-16958248.41
10,-16243585.27
2,-13781168.82


### Insight
Store 38 recorded the largest sales increase between 2010 and 2012, while Store 14 experienced the largest decline. Only three stores showed positive sales changes, indicating that most stores had lower total sales in 2012 than in 2010.

Question 14: Department Sales Trends Over Time.

In [ ]:
dept_year_sales = sales.groupby(['Dept','Year'])['Weekly_Sales'].sum().unstack()
pd.set_option('display.max_rows',None)
pd.set_option('display.max_columns',None)
dept_year_sales

Year,2010,2011,2012
Dept,,,
1,4.284353e+07,4.534931e+07,3.544593e+07
2,9.405293e+07,1.015053e+08,8.505299e+07
3,2.509114e+07,2.701822e+07,2.378310e+07
4,5.557729e+07,6.056611e+07,5.100335e+07
5,5.177156e+07,5.036340e+07,3.347240e+07
6,1.237954e+07,1.008493e+07,5.956193e+06
7,5.842404e+07,5.920394e+07,3.784958e+07
8,6.536836e+07,7.003633e+07,5.887608e+07
9,4.312974e+07,4.508026e+07,4.018326e+07


### Insight

Many departments recorded higher total sales in 2011 than in 2010, followed by lower sales in 2012. Department 92 consistently recorded the highest total sales across all three years. Some departments showed unusually low or negative sales values, which may require further investigation.

Question 15 — Department Sales Change from 2010 to 2012

In [ ]:
dept_sales_change = (dept_year_sales[2012] - dept_year_sales[2010]).rename('sales_change')
# Departments with the largest sales increases
print("Top 5 Departments by Sales Increase:")
print(dept_sales_change.sort_values(ascending=False).head())

# Departments with the largest sales decreases
print("\nTop 5 Departments by Sales Decrease:")
print(dept_sales_change.sort_values().head())

Top 5 Departments by Sales Increase:
Dept
87    3776524.49
48     417173.51
99      94721.96
96      55262.71
77       8546.89
Name: sales_change, dtype: float64

Top 5 Departments by Sales Decrease:
Dept
72   -30201479.42
7    -20574458.69
5    -18299165.09
95   -11966649.67
55   -11342832.95
Name: sales_change, dtype: float64


### Insight

Department 87 recorded the largest sales increase between 2010 and 2012, while Department 72 experienced the largest decline. Although a few departments showed positive sales growth, several departments experienced substantial decreases. These differences highlight the need to investigate department-level sales trends and identify possible reasons for declining performance.

Question 16: Identify Departments with Consistent Sales Growth

In [ ]:
consistant_growth = dept_year_sales[
    (dept_year_sales[2011]>dept_year_sales[2010]) &
     (dept_year_sales[2012]>dept_year_sales[2011])
]
consistant_growth

Year,2010,2011,2012
Dept,,,
48,569338.13,786954.84,986511.64


### Insight

Department 48 was the only department identified by this analysis with consistent sales growth from 2010 through 2012. Its total sales increased in both consecutive periods, making it a useful candidate for further investigation into factors associated with its growth.